Group F aka Humongous data Project 1 notebook

In [1]:
# Spark initialisation
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

sc = SparkContext('local', 'Project1_notebook')

spark = (
    SparkSession.builder
    .appName('Project1_notebook')
    .enableHiveSupport()   # persist tables to local Hive metastore
    .getOrCreate()
)

In [2]:
from pathlib import Path
INBOX_PATH = Path("data/inbox")
OUTBOX_PATH = Path("data/outbox")
MANIFEST_PATH = Path("state/manifest.json")

In [3]:
# Check input directory and read in required files based on what is new / changed since last time
import os
import json
import pyarrow.parquet as pq

# Load manifest
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
else:
    manifest = {"processed_files": []}

processed_files = {f["filename"]: f for f in manifest["processed_files"]}
file_paths_to_process = []

# Process files in inbox
for file_path in Path("data/inbox").glob("*_tripdata_*"):
    file_name = str(file_path)
    
    if file_name in processed_files:
        if processed_files[file_name]["size"] == file_path.stat().st_size:
            if  processed_files[file_name]["row_count"] ==  pq.ParquetFile(file_path).metadata.num_rows:
                print(f"Skipping {file_name}, already processed.")
                continue
    
    print("Processing file: " + str(file_path))
    
    if file_name not in processed_files:
        processed_files[file_name] = {"filename": file_name}
    processed_files[file_name]["status"] = "processing"
    file_paths_to_process.append(file_path)

manifest["processed_files"] = list(processed_files.values())
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

KeyError: 'size'

In [ ]:
# Ingestion TODO 
# filter useful columns


In [195]:
# Transformations - parse and cast types correctly

from pyspark.sql.types import *

trip_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", BinaryType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
])

zone_schema = StructType([
    StructField("LocationID", LongType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

trips_df = spark.read.schema(trip_schema).parquet("data/inbox/yellow_tripdata_2025-01.parquet") \
    .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1)) \
    .withColumn("ingested_at", F.current_timestamp())
zones_df = spark.read.schema(zone_schema).parquet("data/inbox/taxi_zone_lookup.parquet")

t1_count = trips_df.count()
z1_count = zones_df.count()

In [196]:
# Transformations - apply data cleaning rules

# Firstly add two rows 'trip_duration_minutes' and 'avg_speed_kmh' to help with cleaning the data
trips_new_df = trips_df \
    .withColumn("trip_duration_minutes", ((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60).cast("int")) \
    .withColumn("avg_speed_kmh", (F.col("trip_distance") * 1.60934) / (F.col("trip_duration_minutes") / 60))

# Clean data
trips_cleaned = (trips_new_df
    .filter(F.col("passenger_count").isNotNull() & (F.col("passenger_count") > 0))               # Passenger count cannot be null or less/equal than 0
    .filter(F.col("trip_distance") >= 0)                                                         # Trip distance cannot be less than 0
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))                      # Trip end time cannot be before trip start time
    .filter(F.col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99]))                                    # According to NYC TLC data dictionary Ratecode ID can be from 1-6 or 99 (https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
    .filter(F.col("payment_type").isin([0, 1, 2, 3, 4, 5, 6]))                                   # According to NYC TLC data dictionary payment type can be from 0-6
    .filter((F.col("trip_duration_minutes") > 0) & (F.col("trip_duration_minutes") < 1440))      # The trip shouldn't realistically be less/equal than 0 minutes or more than 1 day
    .filter((F.col("avg_speed_kmh") >= 2) & (F.col("avg_speed_kmh") <= 130)))                    # The average speed shouldn't be less than 2km/h or more than 130km/h

zones_cleaned = (zones_df  
    .filter(F.col("LocationID").isNotNull() & (F.col("LocationID") > 0))                         # location ID shouldn't be null
    .filter(F.col("Borough").isNotNull() & (F.col("Borough") != ""))                             # Borough shouldn't be null
    .filter(F.col("Zone").isNotNull() & (F.col("Zone") != "")))                                  # Zone shouldn't be null

t2_count = trips_cleaned.count()
z2_count = zones_cleaned.count()

In [197]:
# Transformations - deduplicate records using a defined key

trips_cleaned = trips_cleaned.dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"])    # The same vendor couldn't realistically start a ride from the same location at the same time
zones_cleaned = zones_df.dropDuplicates(["LocationID"])                                               # Zones shouldn't have the same ID

t3_count = trips_cleaned.count()
z3_count = zones_cleaned.count()

In [207]:
pickup_after_dropoff_df = trips_df.filter((F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"))).sort(F.col("trip_distance").desc())
longest_rides_df = trips_new_df.filter(F.col("trip_duration_minutes") >= 1440).sort((F.col("trip_duration_minutes").desc()))
fastest_rides_df = trips_new_df.filter((F.col("trip_duration_minutes") > 0) & (F.col("avg_speed_kmh") > 130)).sort((F.col("avg_speed_kmh").desc()))

pickup_after_dropoff_df.select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "passenger_count").show(5)
print(f"Number of rows where pickup datetime is later than dropoff datetime: {pickup_after_dropoff_df.count()}.\n")

longest_rides_df.select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "passenger_count", "trip_duration_minutes").show(5)
print(f"Number of rows where ride is longer than 24 hours: {longest_rides_df.count()}.\n")

fastest_rides_df.select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "trip_duration_minutes", "avg_speed_kmh").show(5)
print(f"Number of rows where ride's average speed is more than 130km/h: {fastest_rides_df.count()}.\n")

print(f"Trips: {t1_count} → {t2_count} → {t3_count} (Cleaning removed {t1_count - t2_count} rows and deduplicating removed {t2_count - t3_count} rows).")
print(f"Zones: {z1_count} → {z2_count} → {z3_count} (Cleaning removed {z1_count - z2_count} rows and deduplicating removed {z2_count - z3_count} rows).")

+--------+--------------------+---------------------+-------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|passenger_count|
+--------+--------------------+---------------------+-------------+---------------+
|       6| 2025-01-06 11:01:53|  2025-01-06 11:01:23|         20.8|           NULL|
|       6| 2025-01-01 10:01:27|  2025-01-01 10:01:00|        19.19|           NULL|
|       6| 2025-01-21 00:01:52|  2025-01-21 00:01:36|        18.41|           NULL|
|       6| 2025-01-14 11:01:44|  2025-01-14 11:01:43|        17.88|           NULL|
|       6| 2025-01-12 22:01:06|  2025-01-12 22:01:05|        17.35|           NULL|
+--------+--------------------+---------------------+-------------+---------------+
only showing top 5 rows
Number of rows where pickup datetime is later than dropoff datetime: 124.

+--------+--------------------+---------------------+-------------+---------------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dro

In [200]:
# Enrichment
# logic for creating required derived columns: trip_duration_minutes, pickup_date

clean_df = (trips_cleaned
    .withColumn("pickup_date", F.date_format("tpep_pickup_datetime", "MMMM dd, yyyy"))
)

In [201]:
from pyspark.sql.functions import broadcast

enriched_df = clean_df \
    .join(broadcast(zones_cleaned.selectExpr("LocationID as PULocationID", "Zone as pickup_zone")), "PULocationID", "left") \
    .join(broadcast(zones_cleaned.selectExpr("LocationID as DOLocationID", "Zone as dropoff_zone")), "DOLocationID", "left")

enriched_df.show(5)

+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+--------------------+---------------------+------------------+----------------+--------------------+--------------------+
|DOLocationID|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|         source_file|         ingested_at|trip_duration_minutes|     avg_speed_kmh|     pickup_date|         pickup_zone|        dropoff_zone|
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+----------------

In [202]:
# Scenario
# Add a boolean column is_peak_hour to the output. Peak = Monday-Friday, 07:00-09:00 or 16:00-19:00 (local time).

enriched_df = enriched_df \
    .withColumn("pickup_local", F.from_utc_timestamp(F.col("tpep_pickup_datetime"), "America/New_York")) \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("is_peak_hour",
        (
            (F.col("pickup_dayofweek").between(2, 6)) &
            ((F.col("pickup_hour").between(7, 8)) | (F.col("pickup_hour").between(16, 18)))
        )
    ) \
    .drop("pickup_local", "pickup_hour", "pickup_dayofweek")

inside_peak_count = enriched_df.filter(F.col("is_peak_hour") == "true").count()
outside_peak_count = enriched_df.filter(F.col("is_peak_hour") == "false").count()

print(f"Trips started inside peak hours - {inside_peak_count} - and trips started outside peak hours - {outside_peak_count}.")

Trips started inside peak hours - 609752 - and trips started outside peak hours - 2150478.


In [203]:
# Output - create output

output_df = enriched_df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "pickup_zone", "dropoff_zone", "passenger_count", "trip_distance", "trip_duration_minutes", "pickup_date", "is_peak_hour", "source_file", "ingested_at")
output_df.show(10)

+--------------------+---------------------+------------+------------+--------------------+--------------------+---------------+-------------+---------------------+----------------+------------+--------------------+--------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|         pickup_zone|        dropoff_zone|passenger_count|trip_distance|trip_duration_minutes|     pickup_date|is_peak_hour|         source_file|         ingested_at|
+--------------------+---------------------+------------+------------+--------------------+--------------------+---------------+-------------+---------------------+----------------+------------+--------------------+--------------------+
| 2025-01-01 00:00:31|  2025-01-01 00:08:31|         234|         137|            Union Sq|            Kips Bay|              1|          1.0|                    8|January 01, 2025|       false|yellow_tripdata_2...|2026-03-08 13:15:...|
| 2025-01-01 00:01:06|  2025-01-01 00:09:41|        

In [205]:
# Output - write into output file

output_df.write \
    .mode("append") \
    .parquet("data/outbox/trips_enriched.parquet")

In [206]:
test = spark.read.parquet("data/outbox/trips_enriched.parquet")
test = test.sort((F.col("ingested_at").desc()))
test_count = test.count()
test.show()
print(f"The output file contains {test_count} rows.")

+--------------------+---------------------+------------+------------+--------------------+--------------------+---------------+-------------+---------------------+----------------+------------+--------------------+--------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|         pickup_zone|        dropoff_zone|passenger_count|trip_distance|trip_duration_minutes|     pickup_date|is_peak_hour|         source_file|         ingested_at|
+--------------------+---------------------+------------+------------+--------------------+--------------------+---------------+-------------+---------------------+----------------+------------+--------------------+--------------------+
| 2025-01-01 00:00:31|  2025-01-01 00:08:31|         234|         137|            Union Sq|            Kips Bay|              1|          1.0|                    8|January 01, 2025|       false|yellow_tripdata_2...|2026-03-08 13:16:...|
| 2025-01-01 00:00:00|  2025-01-01 01:03:09|        

In [ ]:
# mapping, filter, 